In [33]:

import requests
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

eta_folder = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)
eta_folder.mkdir(parents=True, exist_ok=True)

prototype_file = eta_folder / "eta_prototype_results.csv"
final_file = eta_folder / "eta_dataset_final.csv"

print("ETA folder ready:", eta_folder)


ETA folder ready: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model


In [34]:

prototype_columns = [
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "routing_provider",
    "record_type",
]

def estimate_eta(
    request_latitude,
    request_longitude,
    rescue_latitude,
    rescue_longitude,
    request_id
):
    request_latitude = float(request_latitude)
    request_longitude = float(request_longitude)
    rescue_latitude = float(rescue_latitude)
    rescue_longitude = float(rescue_longitude)

    if not request_id:
        raise ValueError("request_id is required.")

    for lat, lon in [
        (request_latitude, request_longitude),
        (rescue_latitude, rescue_longitude)
    ]:
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            raise ValueError("Invalid GPS coordinates.")

    url = (
        "https://router.project-osrm.org/route/v1/driving/"
        f"{rescue_longitude},{rescue_latitude};"
        f"{request_longitude},{request_latitude}"
    )

    response = requests.get(
        url,
        params={"overview": "false"},
        timeout=30
    )
    response.raise_for_status()

    data = response.json()

    if data.get("code") != "Ok" or not data.get("routes"):
        raise ValueError("No drivable route was found.")

    route = data["routes"][0]
    distance_km = route["distance"] / 1000
    travel_minutes = route["duration"] / 60

    # Prototype timestamp in the computer's local timezone
    estimated_arrival = (
        datetime.now().astimezone()
        + timedelta(seconds=route["duration"])
    )

    result = {
        "request_id": str(request_id),
        "request_latitude": request_latitude,
        "request_longitude": request_longitude,
        "rescue_team_latitude": rescue_latitude,
        "rescue_team_longitude": rescue_longitude,
        "route_distance_km": round(distance_km, 2),
        "estimated_travel_minutes": round(travel_minutes, 1),
        "estimated_arrival_time": estimated_arrival.isoformat(
            timespec="minutes"
        ),
        "routing_provider": "OSRM",
        "record_type": "prototype_estimate",
    }

    # Initialize a clean CSV if it does not exist or is empty.
    if (
        not prototype_file.exists()
        or prototype_file.stat().st_size == 0
    ):
        pd.DataFrame(columns=prototype_columns).to_csv(
            prototype_file, index=False
        )

    # Read and validate the existing CSV before appending.
    existing = pd.read_csv(prototype_file)

    if list(existing.columns) != prototype_columns:
        raise ValueError(
            "Prototype CSV schema is inconsistent. "
            "Back up and remove the prototype CSV, then rerun this cell."
        )

    # Update an existing request ID rather than creating duplicates.
    existing = existing[
        existing["request_id"].astype(str) != str(request_id)
    ]

    updated = pd.concat(
        [existing, pd.DataFrame([result])],
        ignore_index=True
    )

    updated.to_csv(prototype_file, index=False)

    return result


In [35]:
print("ETA function defined:", callable(estimate_eta))

ETA function defined: True


In [36]:
result = estimate_eta(
    request_latitude=28.6139,
    request_longitude=77.2090,
    rescue_latitude=28.6200,
    rescue_longitude=77.2150,
    request_id="TEST-001"
)

print("ETA result:")
print(result)

ParserError: Error tokenizing data. C error: Expected 5 fields in line 4, saw 10


In [37]:

from pathlib import Path
import pandas as pd
from datetime import datetime
import shutil

eta_folder = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

prototype_file = eta_folder / "eta_prototype_results.csv"

prototype_columns = [
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "routing_provider",
    "record_type",
]

# Back up the existing file before replacing it
if prototype_file.exists():
    backup_file = eta_folder / (
        "eta_prototype_backup_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + ".csv"
    )
    shutil.copy2(prototype_file, backup_file)
    print("Backup created:", backup_file)

# Recreate a clean CSV with consistent headers
pd.DataFrame(columns=prototype_columns).to_csv(
    prototype_file,
    index=False
)

print("Prototype CSV repaired:", prototype_file)
print("Columns:", len(prototype_columns))


Backup created: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_prototype_backup_20261020_013839.csv
Prototype CSV repaired: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_prototype_results.csv
Columns: 10


In [38]:

result = estimate_eta(
    request_latitude=28.6139,
    request_longitude=77.2090,
    rescue_latitude=28.6200,
    rescue_longitude=77.2150,
    request_id="TEST-001"
)

print("ETA result:")
print(result)


ETA result:
{'request_id': 'TEST-001', 'request_latitude': 28.6139, 'request_longitude': 77.209, 'rescue_team_latitude': 28.62, 'rescue_team_longitude': 77.215, 'route_distance_km': 3.97, 'estimated_travel_minutes': 7.0, 'estimated_arrival_time': '2026-10-20T01:45+05:30', 'routing_provider': 'OSRM', 'record_type': 'prototype_estimate'}


In [39]:
import pandas as pd

df = pd.read_csv(prototype_file)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate IDs:", df["request_id"].duplicated().sum())

display(df)

Rows: 1
Columns: 10
Duplicate IDs: 0


,request_id,request_latitude,request_longitude,rescue_team_latitude,rescue_team_longitude,route_distance_km,estimated_travel_minutes,estimated_arrival_time,routing_provider,record_type
0,TEST-001,28.6139,77.209,28.62,77.215,3.97,7.0,2026-10-20T01:45+05:30,OSRM,prototype_estimate


In [40]:
import pandas as pd

df = pd.read_csv(prototype_file)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate IDs:", df["request_id"].duplicated().sum())

display(df[[
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "routing_provider",
    "record_type"
]])

Rows: 1
Columns: 10
Duplicate IDs: 0


,request_id,request_latitude,request_longitude,rescue_team_latitude,rescue_team_longitude,route_distance_km,estimated_travel_minutes,estimated_arrival_time,routing_provider,record_type
0,TEST-001,28.6139,77.209,28.62,77.215,3.97,7.0,2026-10-20T01:45+05:30,OSRM,prototype_estimate


In [41]:

import pandas as pd
from pathlib import Path

eta_folder = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

prototype_file = eta_folder / "eta_prototype_results.csv"
final_file = eta_folder / "eta_dataset_final.csv"

# Load the verified prototype dataset
df = pd.read_csv(prototype_file)

# Add columns for future real-trip and flood-context data
future_columns = [
    "dispatch_time",
    "actual_arrival_time",
    "actual_travel_minutes",
    "rainfall",
    "flood_severity",
]

for column in future_columns:
    if column not in df.columns:
        df[column] = pd.NA

# Arrange the final dataset schema
final_columns = [
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "dispatch_time",
    "actual_arrival_time",
    "actual_travel_minutes",
    "rainfall",
    "flood_severity",
    "routing_provider",
    "record_type",
]

df = df[final_columns]
df = df.drop_duplicates(subset=["request_id"], keep="last")

# Save final CSV
df.to_csv(final_file, index=False)

# Verify the saved file
check = pd.read_csv(final_file)

print("FINAL ETA DATASET CREATED")
print("File:", final_file)
print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Duplicate IDs:", check["request_id"].duplicated().sum())
print("Estimated travel times:", check["estimated_travel_minutes"].notna().sum())
print("Actual travel-time labels:", check["actual_travel_minutes"].notna().sum())

display(check)


FINAL ETA DATASET CREATED
File: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv
Rows: 1
Columns: 15
Duplicate IDs: 0
Estimated travel times: 1
Actual travel-time labels: 0


,request_id,request_latitude,request_longitude,rescue_team_latitude,rescue_team_longitude,route_distance_km,estimated_travel_minutes,estimated_arrival_time,dispatch_time,actual_arrival_time,actual_travel_minutes,rainfall,flood_severity,routing_provider,record_type
0,TEST-001,28.6139,77.209,28.62,77.215,3.97,7.0,2026-10-20T01:45+05:30,NaN,NaN,NaN,NaN,NaN,OSRM,prototype_estimate


In [42]:

from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
base_path = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data"
)

eta_folder = base_path / "ETA_dataset_model"
eta_folder.mkdir(parents=True, exist_ok=True)

flood_file = (
    base_path / "indoflood_processed_Data"
    / "floodevents_indofloods_processed.csv"
)

rainfall_file = (
    base_path / "indoflood_processed_Data"
    / "precipitation_variables_indofloods_processed.csv"
)

catchment_file = (
    base_path / "indoflood_processed_Data"
    / "catchment_characteristics_indofloods_processed.csv"
)

prototype_file = eta_folder / "eta_prototype_results.csv"

context_output = eta_folder / "eta_flood_context_dataset.csv"
final_output = eta_folder / "eta_dataset_final.csv"

# --------------------------------------------------
# 2. Load historical flood and rainfall datasets
# --------------------------------------------------
flood = pd.read_csv(flood_file, low_memory=False)
rainfall = pd.read_csv(rainfall_file, low_memory=False)

# EventID is the common key in these two files.
if flood["EventID"].duplicated().any():
    raise ValueError(
        "Flood dataset contains duplicate EventID values. "
        "Inspect them before joining."
    )

if rainfall["EventID"].duplicated().any():
    raise ValueError(
        "Rainfall dataset contains duplicate EventID values. "
        "Inspect them before joining."
    )

context = flood.merge(
    rainfall,
    on="EventID",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Flood/rainfall join:")
print(context["_merge"].value_counts())

context.drop(columns="_merge", inplace=True)

# --------------------------------------------------
# 3. Add regional catchment context only if safely linkable
# --------------------------------------------------
catchment = pd.read_csv(catchment_file, low_memory=False)

# EventID and GaugeID are not assumed to be equivalent.
# Therefore, do not join these tables without a documented mapping.

context["context_link_status"] = (
    "flood_event_and_rainfall_joined_by_EventID"
)

# Keep the historical context dataset separate.
context.to_csv(context_output, index=False)

# --------------------------------------------------
# 4. Load actual route estimates, if available
# --------------------------------------------------
if prototype_file.exists():
    prototype = pd.read_csv(prototype_file, low_memory=False)
else:
    prototype = pd.DataFrame()

prototype_columns = [
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "routing_provider",
    "record_type",
]

for column in prototype_columns:
    if column not in prototype.columns:
        prototype[column] = pd.NA

prototype = prototype[prototype_columns].copy()

if not prototype.empty:
    prototype = prototype.dropna(subset=["request_id"])
    prototype["request_id"] = prototype["request_id"].astype(str).str.strip()
    prototype = prototype[prototype["request_id"] != ""]
    prototype = prototype.drop_duplicates(
        subset=["request_id"], keep="last"
    )

    # Ensure existing estimates are marked as estimates.
    prototype["record_type"] = "prototype_estimate"

# --------------------------------------------------
# 5. Add future real-trip fields without inventing values
# --------------------------------------------------
final_columns = [
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "dispatch_time",
    "actual_arrival_time",
    "actual_travel_minutes",
    "rainfall",
    "flood_severity",
    "routing_provider",
    "record_type",
]

for column in final_columns:
    if column not in prototype.columns:
        prototype[column] = pd.NA

final_dataset = prototype[final_columns].copy()

final_dataset.to_csv(final_output, index=False)

# --------------------------------------------------
# 6. Verify outputs
# --------------------------------------------------
saved_context = pd.read_csv(context_output, low_memory=False)
saved_final = pd.read_csv(final_output, low_memory=False)

print("\nFINAL OUTPUTS CREATED")
print("=" * 50)

print("\nFlood context dataset:")
print("File:", context_output)
print("Rows:", len(saved_context))
print("Columns:", len(saved_context.columns))

print("\nETA prototype dataset:")
print("File:", final_output)
print("Rows:", len(saved_final))
print("Columns:", len(saved_final.columns))
print(
    "Duplicate request IDs:",
    saved_final["request_id"].duplicated().sum()
)
print(
    "Records with estimated travel time:",
    saved_final["estimated_travel_minutes"].notna().sum()
)
print(
    "Records with actual travel time:",
    saved_final["actual_travel_minutes"].notna().sum()
)
print("Model training: Not performed")

display(saved_final.head())


Flood/rainfall join:
_merge
both          4548
left_only        0
right_only       0
Name: count, dtype: int64

FINAL OUTPUTS CREATED

Flood context dataset:
File: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_flood_context_dataset.csv
Rows: 4548
Columns: 24

ETA prototype dataset:
File: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv
Rows: 1
Columns: 15
Duplicate request IDs: 0
Records with estimated travel time: 1
Records with actual travel time: 0
Model training: Not performed


,request_id,request_latitude,request_longitude,rescue_team_latitude,rescue_team_longitude,route_distance_km,estimated_travel_minutes,estimated_arrival_time,dispatch_time,actual_arrival_time,actual_travel_minutes,rainfall,flood_severity,routing_provider,record_type
0,TEST-001,28.6139,77.209,28.62,77.215,3.97,7.0,2026-10-20T01:45+05:30,NaN,NaN,NaN,NaN,NaN,OSRM,prototype_estimate


In [43]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings

# ============================================================
# JEEVAN LINK — ETA DATASET INTEGRATION
# Sources: IFI, IndoFloods, Disaster/SOS, Assistance
# Output: ETA_dataset_model
# This prepares data; it does NOT train an ETA model.
# ============================================================

# 1. PROJECT PATHS
BASE_DIR = Path(r"C:\Users\ishit\jeevan_link_ml")
DATA_DIR = BASE_DIR / "Data" / "Processed_data"
ETA_DIR = DATA_DIR / "ETA_dataset_model"
ETA_DIR.mkdir(parents=True, exist_ok=True)

# 2. FIND DATASETS RECURSIVELY
def find_dataset(filename, required=False):
    matches = list(DATA_DIR.rglob(filename))

    # Avoid accidentally reading files from the ETA output folder.
    matches = [
        p for p in matches
        if ETA_DIR not in p.parents
    ]

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not find {filename} under {DATA_DIR}"
            )
        print(f"WARNING: Dataset not found: {filename}")
        return None

    if len(matches) > 1:
        print(f"\nMultiple copies found for {filename}:")
        for i, path in enumerate(matches, 1):
            print(f"  {i}. {path}")
        print("Using the first match:", matches[0])

    return matches[0]


# 3. LOAD CSV SAFELY
def load_csv(filename, required=False):
    path = find_dataset(filename, required=required)

    if path is None:
        return None

    df = pd.read_csv(path, low_memory=False)
    df.columns = (
        df.columns.astype(str)
        .str.strip()
    )

    print(f"\nLoaded: {filename}")
    print(f"Path: {path}")
    print(f"Shape: {df.shape}")

    return df


# 4. LOAD THE PREVIOUSLY USED DATASETS
ifi = load_csv(
    "IFI_flood_events_processed.csv"
)

indoflood_events = load_csv(
    "floodevents_indofloods_processed.csv"
)

indoflood_rain = load_csv(
    "precipitation_variables_indofloods_processed.csv"
)

indoflood_catchment = load_csv(
    "catchment_characteristics_indofloods_processed.csv"
)

district_impact = load_csv(
    "District_FloodImpact_processed.csv"
)

sos = load_csv(
    "sos_urgency_dataset_processed.csv"
)

assistance = load_csv(
    "assistance_model_dataset_final_cleaned.csv"
)


# 5. HELPER: ADD SOURCE AND ROW IDENTIFIERS
def add_source_metadata(df, source_name):
    if df is None:
        return None

    df = df.copy()
    df["source_dataset"] = source_name
    df["source_row_id"] = np.arange(1, len(df) + 1)

    return df


ifi = add_source_metadata(ifi, "IFI")
indoflood_events = add_source_metadata(
    indoflood_events, "IndoFloods_Events"
)
indoflood_rain = add_source_metadata(
    indoflood_rain, "IndoFloods_Precipitation"
)
indoflood_catchment = add_source_metadata(
    indoflood_catchment, "IndoFloods_Catchment"
)
district_impact = add_source_metadata(
    district_impact, "District_FloodImpact"
)
sos = add_source_metadata(sos, "Disaster_SOS_Urgency")
assistance = add_source_metadata(
    assistance, "Assistance"
)


# 6. CREATE A STANDARD ETA SCHEMA
# These fields remain missing unless the source contains
# trustworthy corresponding information.
ETA_COLUMNS = [
    "request_id",
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "dispatch_time",
    "actual_arrival_time",
    "actual_travel_minutes",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
    "urgency_label",
    "people_affected",
    "people_trapped",
    "medical_emergency",
    "rescue_required",
    "water_level",
    "building_condition",
    "location_risk",
    "source_dataset",
    "source_row_id",
    "record_type"
]


def normalize_sos_for_eta(df):
    if df is None:
        return pd.DataFrame(columns=ETA_COLUMNS)

    result = pd.DataFrame(index=df.index)

    # Preserve SOS identifiers; don't invent shared IDs.
    if "sos_id" in df.columns:
        result["request_id"] = df["sos_id"].astype("string")
    else:
        result["request_id"] = pd.Series(
            pd.NA, index=df.index, dtype="string"
        )

    # Map fields only when their names exist in the source.
    field_map = {
        "urgency_label": "urgency_label",
        "people_affected": "people_affected",
        "people_trapped": "people_trapped",
        "medical_emergency": "medical_emergency",
        "rescue_required": "rescue_required",
        "water_level": "water_level",
        "building_condition": "building_condition",
        "location_risk": "location_risk",
        "source_dataset": "source_dataset",
        "source_row_id": "source_row_id",
    }

    for target, source in field_map.items():
        if source in df.columns:
            result[target] = df[source]
        else:
            result[target] = pd.NA

    # Coordinates and travel-time labels are not inferred.
    for col in ETA_COLUMNS:
        if col not in result.columns:
            result[col] = pd.NA

    result["record_type"] = "sos_context_not_eta_label"

    return result[ETA_COLUMNS]


# 7. SOS REQUEST/URGENCY RECORDS
eta_sos = normalize_sos_for_eta(sos)

eta_sos.to_csv(
    ETA_DIR / "eta_sos_context.csv",
    index=False
)


# 8. KEEP FLOOD DATASETS AS SEPARATE CONTEXT TABLES
# Do NOT concatenate unrelated tables or join EventID to
# GaugeID without a verified relationship.

context_datasets = {
    "eta_ifi_flood_context.csv": ifi,
    "eta_indoflood_events_context.csv": indoflood_events,
    "eta_indoflood_precipitation_context.csv": indoflood_rain,
    "eta_indoflood_catchment_context.csv": indoflood_catchment,
    "eta_district_flood_impact_context.csv": district_impact,
}

for output_name, df in context_datasets.items():
    if df is not None:
        df.to_csv(ETA_DIR / output_name, index=False)


# 9. KEEP ASSISTANCE LABELS AS A SEPARATE REFERENCE TABLE
# Do not attach assistance rows to SOS rows by row position.
if assistance is not None:
    assistance.to_csv(
        ETA_DIR / "eta_assistance_reference.csv",
        index=False
    )


# 10. CREATE A MASTER INVENTORY OF ALL SOURCE TABLES
inventory = []

all_sources = {
    "IFI": ifi,
    "IndoFloods Events": indoflood_events,
    "IndoFloods Precipitation": indoflood_rain,
    "IndoFloods Catchment": indoflood_catchment,
    "District Flood Impact": district_impact,
    "Disaster SOS Urgency": sos,
    "Assistance": assistance,
}

for source_name, df in all_sources.items():
    if df is not None:
        inventory.append({
            "source_dataset": source_name,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": json.dumps(
                list(df.columns), ensure_ascii=False
            )
        })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(
    ETA_DIR / "eta_source_inventory.csv",
    index=False
)


# 11. CREATE A CONSOLIDATED ETA CONTEXT DATASET
# Each row represents an SOS context record.
# Flood tables are retained separately until a verified
# common key or geographic/time matching rule is available.
eta_final = eta_sos.copy()

eta_final.to_csv(
    ETA_DIR / "eta_dataset_final.csv",
    index=False
)


# 12. DATA QUALITY REPORT
print("\n" + "=" * 60)
print("JEEVAN LINK ETA DATA PREPARATION COMPLETE")
print("=" * 60)

print("\nSource inventory:")
if not inventory_df.empty:
    print(
        inventory_df[
            ["source_dataset", "rows", "columns"]
        ].to_string(index=False)
    )
else:
    print("No source datasets were found.")

print("\nETA context dataset shape:", eta_final.shape)
print("Saved to:", ETA_DIR / "eta_dataset_final.csv")

if not eta_final.empty:
    print("\nMissing ETA-critical fields:")
    critical_fields = [
        "request_latitude",
        "request_longitude",
        "rescue_team_latitude",
        "rescue_team_longitude",
        "dispatch_time",
        "actual_arrival_time",
        "actual_travel_minutes",
    ]

    for col in critical_fields:
        if col in eta_final.columns:
            print(f"{col}: {eta_final[col].isna().sum()} missing")

print("\nGenerated files:")
for path in sorted(ETA_DIR.glob("eta_*.csv")):
    print(f"- {path.name}")

print(
    "\nIMPORTANT: This is a context dataset, not a trained ETA "
    "dataset. Actual rescue travel-time labels and valid "
    "location/time links are required to train and evaluate "
    "an ETA prediction model."
)



Loaded: IFI_flood_events_processed.csv
Path: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ifi_processed_Data\IFI_flood_events_processed.csv
Shape: (6876, 17)

Loaded: floodevents_indofloods_processed.csv
Path: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\indoflood_processed_Data\floodevents_indofloods_processed.csv
Shape: (4548, 13)

Loaded: precipitation_variables_indofloods_processed.csv
Path: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\indoflood_processed_Data\precipitation_variables_indofloods_processed.csv
Shape: (4548, 11)

Loaded: catchment_characteristics_indofloods_processed.csv
Path: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\indoflood_processed_Data\catchment_characteristics_indofloods_processed.csv
Shape: (155, 103)

Loaded: District_FloodImpact_processed.csv
Path: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ifi_processed_Data\District_FloodImpact_processed.csv
Shape: (732, 5)

Multiple copies found for sos_urgency_dataset_processed.csv:
  1. C

In [45]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data")

sos_path = (
    BASE_DIR
    / "sos_urgency_dataset_model"
    / "sos_urgency_dataset_processed.csv"
)

sos = pd.read_csv(sos_path, low_memory=False)

assert len(sos) > 0, "SOS dataset is empty!"

print("SOS dataset loaded successfully:", sos.shape)
display(sos.head())



SOS dataset loaded successfully: (5016, 40)


,sos_id,sos_description,people_affected,people_trapped,injuries_reported,medical_emergency,water_level,building_condition,rescue_required,food_required,...,disaster_response_message_context,source_medical_help,source_search_and_rescue,source_water,source_food,source_shelter,source_clothing,source_transport,source_buildings,source_floods
0,SOS0001,Four people trapped on a roof as water rises r...,4,Yes,No,No,Rapidly_Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SOS0002,An elderly person is unconscious in a flooded ...,1,Yes,Yes,Yes,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SOS0003,A child is trapped in fast-moving floodwater.,1,Yes,No,No,Rapidly_Rising,Unknown,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SOS0004,Several people are trapped inside a collapsing...,6,Yes,Yes,No,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SOS0005,Water is entering the ground floor and a famil...,5,No,No,No,Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1]:

# Check the SOS dataset's data quality before building ETA features

print("Dataset shape:", sos.shape)

quality_report = pd.DataFrame({
    "dtype": sos.dtypes.astype(str),
    "non_missing": sos.notna().sum(),
    "missing": sos.isna().sum(),
    "missing_percent": (
        sos.isna().mean() * 100
    ).round(2),
    "unique_values": sos.nunique(dropna=True)
})

display(
    quality_report.sort_values(
        "missing_percent",
        ascending=False
    )
)

print("\nSOS urgency distribution:")
display(sos["urgency_label"].value_counts(dropna=False))

context_cols = [
    col for col in sos.columns
    if col.startswith("context_")
    or col.endswith("_context")
    or col.startswith("source_")
]

print("\nExisting flood/disaster context coverage:")
display(
    quality_report.loc[
        quality_report.index.intersection(context_cols)
    ]
)


NameError: name 'sos' is not defined

In [2]:

from pathlib import Path
import pandas as pd

# 1. Define the project data directory
BASE_DIR = Path(r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data")

# 2. Load the correct SOS dataset
sos_path = (
    BASE_DIR
    / "sos_urgency_dataset_model"
    / "sos_urgency_dataset_processed.csv"
)

if not sos_path.exists():
    raise FileNotFoundError(f"SOS dataset not found: {sos_path}")

sos = pd.read_csv(sos_path, low_memory=False)

print("SOS dataset loaded successfully")
print("Dataset shape:", sos.shape)

# 3. Generate the data-quality report
quality_report = pd.DataFrame({
    "dtype": sos.dtypes.astype(str),
    "non_missing": sos.notna().sum(),
    "missing": sos.isna().sum(),
    "missing_percent": (sos.isna().mean() * 100).round(2),
    "unique_values": sos.nunique(dropna=True)
})

display(
    quality_report.sort_values(
        "missing_percent",
        ascending=False
    )
)

# 4. Check urgency-label distribution
print("\nSOS urgency distribution:")
display(sos["urgency_label"].value_counts(dropna=False))

# 5. Inspect existing flood and disaster context
context_cols = [
    col for col in sos.columns
    if col.startswith("context_")
    or col.endswith("_context")
    or col.startswith("source_")
]

print("\nFlood/disaster context data quality:")
display(quality_report.loc[context_cols])

# 6. Verify ETA-critical fields
eta_critical_cols = [
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "dispatch_time",
    "actual_arrival_time",
    "actual_travel_minutes"
]

print("\nETA-critical fields:")
for col in eta_critical_cols:
    if col in sos.columns:
        print(f"{col}: present in source")
    else:
        print(f"{col}: not present in source")


SOS dataset loaded successfully
Dataset shape: (5016, 40)


,dtype,non_missing,missing,missing_percent,unique_values
context_dfsi,float64,4992,24,0.48,743
context_district_fatalities,float64,5000,16,0.32,234
context_event_duration_days,float64,5000,16,0.32,72
indoflood_event_id_context,str,5000,16,0.32,3031
context_district_injuries,float64,5000,16,0.32,85
ifi_impact_district_context,str,5000,16,0.32,725
context_rainfall_t1d,float64,5000,16,0.32,2951
context_peak_flood_level_m,float64,5000,16,0.32,2318
source_clothing,float64,5000,16,0.32,2
source_water,float64,5000,16,0.32,2



SOS urgency distribution:


urgency_label
High        2375
Medium      1305
Critical    1204
Low          132
Name: count, dtype: int64


Flood/disaster context data quality:


,dtype,non_missing,missing,missing_percent,unique_values
context_link_status,str,5016,0,0.00,2
indoflood_event_id_context,str,5000,16,0.32,3031
context_peak_flood_level_m,float64,5000,16,0.32,2318
context_event_duration_days,float64,5000,16,0.32,72
context_rainfall_t1d,float64,5000,16,0.32,2951
ifi_impact_district_context,str,5000,16,0.32,725
context_district_fatalities,float64,5000,16,0.32,234
context_district_injuries,float64,5000,16,0.32,85
context_district_population,float64,5000,16,0.32,731
ifi_area_district_context,str,5000,16,0.32,726



ETA-critical fields:
request_latitude: not present in source
request_longitude: not present in source
rescue_team_latitude: not present in source
rescue_team_longitude: not present in source
dispatch_time: not present in source
actual_arrival_time: not present in source
actual_travel_minutes: not present in source


In [3]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data"
)

files = {
    "IFI": DATA_DIR / "ifi_processed_Data" /
           "IFI_flood_events_processed.csv",

    "IndoFloods Events": DATA_DIR / "indoflood_processed_Data" /
           "floodevents_indofloods_processed.csv",

    "IndoFloods Rainfall": DATA_DIR / "indoflood_processed_Data" /
           "precipitation_variables_indofloods_processed.csv",

    "IndoFloods Catchment": DATA_DIR / "indoflood_processed_Data" /
           "catchment_characteristics_indofloods_processed.csv",

    "District Impact": DATA_DIR / "ifi_processed_Data" /
           "District_FloodImpact_processed.csv",

    "Assistance": DATA_DIR / "Assitance_Dataset_model" /
           "assistance_model_dataset_final_cleaned.csv",
}

for name, path in files.items():
    print("\n" + "=" * 75)
    print(name)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    df = pd.read_csv(path, low_memory=False)

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    # Show likely identifiers, date fields and geographic fields.
    relevant = [
        c for c in df.columns
        if any(term in c.lower() for term in [
            "id", "date", "time", "district", "state",
            "city", "latitude", "longitude", "lat", "lon",
            "location", "gauge", "event", "rain", "flood"
        ])
    ]

    print("\nPotential linking columns:")
    print(relevant)

    display(df[relevant].head(3) if relevant else df.head(3))



IFI
Shape: (6876, 17)
Columns:
['Unnamed: 0', 'UEI', 'Start Date', 'End Date', 'Duration(Days)', 'Main Cause', 'Districts', 'State', 'Human fatality', 'Human injured', 'Human Displaced', 'Animal Fatality', 'Description of Casualties/injured', 'Extent of damage', 'Event Source', 'District_LGD_Codes', 'State_Codes']

Potential linking columns:
['Start Date', 'End Date', 'Districts', 'State', 'Event Source', 'District_LGD_Codes', 'State_Codes']


,Start Date,End Date,Districts,State,Event Source,District_LGD_Codes,State_Codes
0,1967-07-02,1967-07-08,NaN,Assam,IMD,NaN,18
1,1967-07-22,1967-07-28,NaN,"Maharashtra, Gujarat",IMD,NaN,"27,24"
2,1967-08-01,1967-08-30,NaN,Uttar Pradesh,IMD,NaN,9



IndoFloods Events
Shape: (4548, 13)
Columns:
['EventID', 'Start Date', 'End Date', 'Peak Flood Level (m)', 'Peak FL Date', 'Num Peak FL', 'Peak Discharge Q (cumec)', 'Peak Discharge Date', 'Flood Volume (cumec)', 'Event Duration (days)', 'Time to Peak (days)', 'Recession Time (day)', 'Flood Type']

Potential linking columns:
['EventID', 'Start Date', 'End Date', 'Peak Flood Level (m)', 'Peak FL Date', 'Peak Discharge Date', 'Flood Volume (cumec)', 'Event Duration (days)', 'Time to Peak (days)', 'Recession Time (day)', 'Flood Type']


,EventID,Start Date,End Date,Peak Flood Level (m),Peak FL Date,Peak Discharge Date,Flood Volume (cumec),Event Duration (days),Time to Peak (days),Recession Time (day),Flood Type
0,INDOFLOODS-gauge-1010-1,2010-07-21,2010-07-21,47.95,2010-07-21,NaN,NaN,1,1,1,Flood
1,INDOFLOODS-gauge-1010-2,2016-07-23,2016-07-23,48.05,2016-07-23,NaN,NaN,1,1,1,Flood
2,INDOFLOODS-gauge-1010-3,2016-07-26,2016-07-26,48.00,2016-07-26,NaN,NaN,1,1,1,Flood



IndoFloods Rainfall
Shape: (4548, 11)
Columns:
['EventID', 'T1d', 'T2d', 'T3d', 'T4d', 'T5d', 'T6d', 'T7d', 'T8d', 'T9d', 'T10d']

Potential linking columns:
['EventID']


,EventID
0,INDOFLOODS-gauge-1010-1
1,INDOFLOODS-gauge-1010-2
2,INDOFLOODS-gauge-1010-3


... (2 outputs left out)


District Impact
Shape: (732, 5)
Columns:
['Dist_Name', 'Human_fatality', 'Human_injured', 'Population', 'Mean_Flood_Duration']

Potential linking columns:
['Population', 'Mean_Flood_Duration']


,Population,Mean_Flood_Duration
0,618459,1.0
1,190125,NaN
2,67412,NaN



Assistance
Shape: (23753, 12)
Columns:
['id', 'message', 'genre', 'food', 'water', 'shelter', 'medical_help', 'medical_products', 'search_and_rescue', 'transport', 'source_split', 'source_file']

Potential linking columns:
['id']


,id
0,2
1,7
2,12


In [4]:

from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data"
)
OUT_DIR = DATA_DIR / "ETA_dataset_model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load IndoFloods files
event_path = (
    DATA_DIR / "indoflood_processed_Data"
    / "floodevents_indofloods_processed.csv"
)
rain_path = (
    DATA_DIR / "indoflood_processed_Data"
    / "precipitation_variables_indofloods_processed.csv"
)
catchment_path = (
    DATA_DIR / "indoflood_processed_Data"
    / "catchment_characteristics_indofloods_processed.csv"
)

events = pd.read_csv(event_path, low_memory=False)
rain = pd.read_csv(rain_path, low_memory=False)
catchment = pd.read_csv(catchment_path, low_memory=False)

# 1. Check EventID validity and duplicates
print("Event rows:", len(events))
print("Rainfall rows:", len(rain))
print("Unique event IDs:", events["EventID"].nunique())
print("Unique rainfall IDs:", rain["EventID"].nunique())
print("Duplicate event IDs:", events["EventID"].duplicated().sum())
print("Duplicate rainfall IDs:", rain["EventID"].duplicated().sum())

if events["EventID"].duplicated().any():
    raise ValueError(
        "Duplicate EventID values in flood events. Resolve before joining."
    )

if rain["EventID"].duplicated().any():
    raise ValueError(
        "Duplicate EventID values in rainfall. Resolve before joining."
    )

# 2. Join rainfall to events using the shared EventID
flood_context = events.merge(
    rain,
    on="EventID",
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("", "_rain")
)

print("\nEvent/rainfall join:")
print(flood_context["_merge"].value_counts())
flood_context.drop(columns="_merge", inplace=True)

# 3. Extract gauge IDs from EventID, for example:
# INDOFLOODS-gauge-1010-1 -> INDOFLOODS-gauge-1010
def extract_gauge_id(event_id):
    if pd.isna(event_id):
        return pd.NA

    match = re.match(
        r"^(INDOFLOODS-gauge-\d+)-\d+$",
        str(event_id).strip(),
        flags=re.IGNORECASE
    )
    return match.group(1) if match else pd.NA

flood_context["GaugeID_candidate"] = (
    flood_context["EventID"].map(extract_gauge_id)
)

# Compare candidate IDs against actual catchment GaugeID values
catchment["GaugeID"] = catchment["GaugeID"].astype("string").str.strip()
flood_context["GaugeID_candidate"] = (
    flood_context["GaugeID_candidate"].astype("string")
)

known_gauges = set(catchment["GaugeID"].dropna())
matched_gauges = flood_context["GaugeID_candidate"].isin(known_gauges)

print("\nGauge mapping validation:")
print("Events with extracted gauge IDs:",
      flood_context["GaugeID_candidate"].notna().sum())
print("Events matching a catchment GaugeID:", matched_gauges.sum())
print("Events without a matching gauge:", (~matched_gauges).sum())

# Join catchment features only for validated gauge matches
flood_context = flood_context.merge(
    catchment,
    left_on="GaugeID_candidate",
    right_on="GaugeID",
    how="left",
    validate="many_to_one",
    indicator="catchment_merge",
    suffixes=("", "_catchment")
)

print("\nCatchment join:")
print(flood_context["catchment_merge"].value_counts())
flood_context.drop(columns="catchment_merge", inplace=True)

# 4. Save the linked flood-context dataset
output_path = OUT_DIR / "eta_flood_context_dataset.csv"
flood_context.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("Final shape:", flood_context.shape)
print("\nPreview:")
display(flood_context.head())

# 5. District matching diagnostic — do not join yet
ifi_path = (
    DATA_DIR / "ifi_processed_Data"
    / "IFI_flood_events_processed.csv"
)
district_path = (
    DATA_DIR / "ifi_processed_Data"
    / "District_FloodImpact_processed.csv"
)

ifi = pd.read_csv(ifi_path, low_memory=False)
districts = pd.read_csv(district_path, low_memory=False)

def normalize_name(value):
    if pd.isna(value):
        return ""
    return re.sub(
        r"\s+", " ", str(value).strip().lower()
    )

district_names = set(
    districts["Dist_Name"].map(normalize_name)
)

ifi_district_values = (
    ifi["Districts"].dropna().astype(str).unique()
)

exact_matches = []
for value in ifi_district_values:
    # IFI values may contain multiple comma-separated districts.
    names = [normalize_name(x) for x in value.split(",")]
    for name in names:
        if name:
            exact_matches.append(
                (name, name in district_names)
            )

if exact_matches:
    match_rate = sum(m for _, m in exact_matches) / len(exact_matches)
    print("\nDistrict-name diagnostic:")
    print("Individual district names checked:", len(exact_matches))
    print("Exact match rate:", round(match_rate * 100, 2), "%")
else:
    print("\nNo non-missing IFI district names to compare.")

print(
    "\nDistrict matching is diagnostic only. Review unmatched names, "
    "state boundaries, and multi-district events before joining."
)


Event rows: 4548
Rainfall rows: 4548
Unique event IDs: 4548
Unique rainfall IDs: 4548
Duplicate event IDs: 0
Duplicate rainfall IDs: 0

Event/rainfall join:
_merge
both          4548
left_only        0
right_only       0
Name: count, dtype: int64

Gauge mapping validation:
Events with extracted gauge IDs: 4548
Events matching a catchment GaugeID: 4548
Events without a matching gauge: 0

Catchment join:
catchment_merge
both          4548
left_only        0
right_only       0
Name: count, dtype: int64


C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\2883313991.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  flood_context = flood_context.merge(



Saved: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_flood_context_dataset.csv
Final shape: (4548, 127)

Preview:


,EventID,Start Date,End Date,Peak Flood Level (m),Peak FL Date,Num Peak FL,Peak Discharge Q (cumec),Peak Discharge Date,Flood Volume (cumec),Event Duration (days),...,2015_HDI,2005_HDI,2000_HDI,1995_HDI,1990_HDI,Population Count,Population Density,Land cover,Soil type,lithology type
0,INDOFLOODS-gauge-1010-1,2010-07-21,2010-07-21,47.95,2010-07-21,1,NaN,NaN,NaN,1,...,0.605959,0.549116,0.527775,0.506860,0.488529,209858.27730,20.618854,Forest,Luvisols,No dominant class
1,INDOFLOODS-gauge-1010-2,2016-07-23,2016-07-23,48.05,2016-07-23,1,NaN,NaN,NaN,1,...,0.605959,0.549116,0.527775,0.506860,0.488529,209858.27730,20.618854,Forest,Luvisols,No dominant class
2,INDOFLOODS-gauge-1010-3,2016-07-26,2016-07-26,48.00,2016-07-26,1,NaN,NaN,NaN,1,...,0.605959,0.549116,0.527775,0.506860,0.488529,209858.27730,20.618854,Forest,Luvisols,No dominant class
3,INDOFLOODS-gauge-1010-4,2017-08-11,2017-08-13,48.95,2017-08-12,1,NaN,NaN,NaN,3,...,0.605959,0.549116,0.527775,0.506860,0.488529,209858.27730,20.618854,Forest,Luvisols,No dominant class
4,INDOFLOODS-gauge-1012-1,2010-07-21,2010-07-21,48.10,2010-07-21,1,NaN,NaN,NaN,1,...,0.637121,0.562175,0.529422,0.501011,0.474977,43702.44413,289.760264,No dominant class,Cambisols,No dominant class



District-name diagnostic:
Individual district names checked: 14876
Exact match rate: 88.67 %

District matching is diagnostic only. Review unmatched names, state boundaries, and multi-district events before joining.


In [5]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data"
)
ETA_DIR = DATA_DIR / "ETA_dataset_model"

path = ETA_DIR / "eta_flood_context_dataset.csv"
df = pd.read_csv(path, low_memory=False)

print("Dataset shape:", df.shape)
print("Unique event IDs:", df["EventID"].nunique())

# Check event dates
df["Start Date"] = pd.to_datetime(
    df["Start Date"], errors="coerce"
)
df["End Date"] = pd.to_datetime(
    df["End Date"], errors="coerce"
)

invalid_dates = (
    df["Start Date"].isna()
    | df["End Date"].isna()
    | (df["End Date"] < df["Start Date"])
)

print("\nInvalid or missing event date rows:", invalid_dates.sum())

# Check useful feature coverage
feature_cols = [
    "Peak Flood Level (m)",
    "Event Duration (days)",
    "Time to Peak (days)",
    "T1d", "T2d", "T3d", "T4d", "T5d",
    "T6d", "T7d", "T8d", "T9d", "T10d",
    "Drainage Area",
    "Road Density",
    "Population Density",
]

available = [c for c in feature_cols if c in df.columns]

quality = pd.DataFrame({
    "non_missing": df[available].notna().sum(),
    "missing": df[available].isna().sum(),
    "missing_percent": (
        df[available].isna().mean() * 100
    ).round(2),
})

print("\nFeature completeness:")
display(quality)

print("\nSample records:")
display(df[["EventID", "Start Date", "End Date"] + available[:6]].head())


Dataset shape: (4548, 127)
Unique event IDs: 4548

Invalid or missing event date rows: 0

Feature completeness:


,non_missing,missing,missing_percent
Peak Flood Level (m),4548,0,0.0
Event Duration (days),4548,0,0.0
Time to Peak (days),4548,0,0.0
T1d,4548,0,0.0
T2d,4548,0,0.0
T3d,4548,0,0.0
T4d,4548,0,0.0
T5d,4548,0,0.0
T6d,4548,0,0.0
T7d,4548,0,0.0



Sample records:


,EventID,Start Date,End Date,Peak Flood Level (m),Event Duration (days),Time to Peak (days),T1d,T2d,T3d
0,INDOFLOODS-gauge-1010-1,2010-07-21,2010-07-21,47.95,1,1,32.833274,65.228899,99.413481
1,INDOFLOODS-gauge-1010-2,2016-07-23,2016-07-23,48.05,1,1,27.011286,60.147585,86.528438
2,INDOFLOODS-gauge-1010-3,2016-07-26,2016-07-26,48.00,1,1,50.413312,82.466743,124.859376
3,INDOFLOODS-gauge-1010-4,2017-08-11,2017-08-13,48.95,3,2,46.282306,86.765681,104.849158
4,INDOFLOODS-gauge-1012-1,2010-07-21,2010-07-21,48.10,1,1,86.018135,137.676182,176.841432


In [6]:

from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data"
)

ifi_path = (
    DATA_DIR / "ifi_processed_Data"
    / "IFI_flood_events_processed.csv"
)

district_path = (
    DATA_DIR / "ifi_processed_Data"
    / "District_FloodImpact_processed.csv"
)

ifi = pd.read_csv(ifi_path, low_memory=False)
districts = pd.read_csv(district_path, low_memory=False)

def normalize_district(value):
    if pd.isna(value):
        return ""
    return re.sub(
        r"\s+", " ", str(value).strip().lower()
    )

known_districts = set(
    districts["Dist_Name"]
    .map(normalize_district)
    .loc[lambda s: s != ""]
)

# Split multi-district IFI entries into individual names.
ifi_names = (
    ifi["Districts"]
    .dropna()
    .astype(str)
    .str.split(",")
    .explode()
    .map(normalize_district)
)

ifi_names = ifi_names[ifi_names != ""]

unmatched = sorted(
    set(ifi_names) - known_districts
)

print("Unique IFI district names:", ifi_names.nunique())
print("Unique district-impact names:", len(known_districts))
print("Unmatched unique names:", len(unmatched))
print("\nUnmatched names (first 100):")
display(pd.DataFrame({"unmatched_district": unmatched[:100]}))


Unique IFI district names: 876
Unique district-impact names: 726
Unmatched unique names: 236

Unmatched names (first 100):


,unmatched_district
0,& parts of himachal pradesh
1,& parts of uttar pradesh
2,& various parts of nagaland
3,11 districts
4,14 districts
...,...
95,jalore
96,jaloree
97,janjgir-champa
98,jawad


In [7]:

import re
import pandas as pd

def classify_district_name(name):
    name = str(name).strip().lower()

    if re.search(r"\b\d+\s+districts?\b", name):
        return "MULTIPLE_DISTRICTS_UNSPECIFIED"

    if any(term in name for term in [
        "parts of", "various parts", "various districts",
        "several districts", "whole state", "entire state"
    ]):
        return "REGION_OR_MULTI_DISTRICT"

    if any(term in name for term in [
        "unknown", "not known", "not specified"
    ]):
        return "UNSPECIFIED_LOCATION"

    if name.startswith("&") or name.startswith("and "):
        return "PARTIAL_DESCRIPTION"

    return "POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT"


unmatched_df = pd.DataFrame({
    "unmatched_district": unmatched
})

unmatched_df["classification"] = (
    unmatched_df["unmatched_district"]
    .apply(classify_district_name)
)

print("Classification counts:")
display(
    unmatched_df["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="count")
)

print("\nExamples from each category:")
for category, group in unmatched_df.groupby("classification"):
    print(f"\n{category}")
    display(group.head(10))



Classification counts:


,classification,count
0,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT,203
1,REGION_OR_MULTI_DISTRICT,24
2,MULTIPLE_DISTRICTS_UNSPECIFIED,9



Examples from each category:

MULTIPLE_DISTRICTS_UNSPECIFIED


,unmatched_district,classification
3,11 districts,MULTIPLE_DISTRICTS_UNSPECIFIED
4,14 districts,MULTIPLE_DISTRICTS_UNSPECIFIED
5,14 districts of assam,MULTIPLE_DISTRICTS_UNSPECIFIED
6,19 districts of orissa,MULTIPLE_DISTRICTS_UNSPECIFIED
7,21 districts of assam,MULTIPLE_DISTRICTS_UNSPECIFIED
8,26 districts of rajasthan,MULTIPLE_DISTRICTS_UNSPECIFIED
9,29 districts affected,MULTIPLE_DISTRICTS_UNSPECIFIED
10,3 out of 8 districts,MULTIPLE_DISTRICTS_UNSPECIFIED
11,6 districts,MULTIPLE_DISTRICTS_UNSPECIFIED



POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT


,unmatched_district,classification
12,adjoining areas,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
13,agar-malwa,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
14,ahmedabad,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
15,ahmednagar,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
16,akharpur,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
17,ananthapuramu,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
18,ananthapuramuamu,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
19,arvalli,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
20,ayodhya,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT
21,bagalkotee,POSSIBLE_NAME_VARIANT_OR_UNMATCHED_DISTRICT



REGION_OR_MULTI_DISTRICT


,unmatched_district,classification
0,& parts of himachal pradesh,REGION_OR_MULTI_DISTRICT
1,& parts of uttar pradesh,REGION_OR_MULTI_DISTRICT
2,& various parts of nagaland,REGION_OR_MULTI_DISTRICT
68,different parts of jammu,REGION_OR_MULTI_DISTRICT
92,hyderabad & parts of telangana,REGION_OR_MULTI_DISTRICT
131,many parts of tamil nadu.,REGION_OR_MULTI_DISTRICT
132,many parts of the state,REGION_OR_MULTI_DISTRICT
157,parts of andhra pradesh,REGION_OR_MULTI_DISTRICT
158,parts of himachal pradesh,REGION_OR_MULTI_DISTRICT
159,parts of jammu,REGION_OR_MULTI_DISTRICT


In [8]:

from pathlib import Path
import pandas as pd
import numpy as np

# =========================================================
# JEEVAN LINK: FINAL ETA FEATURE DATASET
# =========================================================

BASE = Path(r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data")
ETA_DIR = BASE / "ETA_dataset_model"

SOS_PATH = (
    BASE / "sos_urgency_dataset_model"
    / "sos_urgency_dataset_processed.csv"
)

FLOOD_PATH = ETA_DIR / "eta_flood_context_dataset.csv"

OUTPUT_PATH = ETA_DIR / "eta_dataset_final.csv"

# 1. Load source datasets
sos = pd.read_csv(SOS_PATH, low_memory=False)
flood = pd.read_csv(FLOOD_PATH, low_memory=False)

print("SOS shape:", sos.shape)
print("Flood context shape:", flood.shape)

# 2. Validate join keys
required_sos = ["sos_id", "indoflood_event_id_context"]
required_flood = ["EventID"]

for col in required_sos:
    if col not in sos.columns:
        raise ValueError(f"Missing SOS column: {col}")

if "EventID" not in flood.columns:
    raise ValueError("Missing EventID in flood context.")

# 3. Normalize join-key formatting
sos["indoflood_event_id_context"] = (
    sos["indoflood_event_id_context"]
    .astype("string")
    .str.strip()
)

flood["EventID"] = (
    flood["EventID"]
    .astype("string")
    .str.strip()
)

# 4. Verify flood-event key uniqueness
if flood["EventID"].duplicated().any():
    raise ValueError(
        "EventID is duplicated in flood context; "
        "resolve duplicates before merging."
    )

# 5. Merge: retain every SOS request
eta = sos.merge(
    flood,
    left_on="indoflood_event_id_context",
    right_on="EventID",
    how="left",
    validate="many_to_one",
    indicator="_flood_join",
    suffixes=("", "_indoflood")
)

print("\nFlood-context matching:")
print(eta["_flood_join"].value_counts())

eta["flood_context_matched"] = (
    eta["_flood_join"] == "both"
).astype("int8")

eta.drop(columns="_flood_join", inplace=True)

# 6. Add standardized ETA fields.
# These remain missing until actual trip data is available.
eta_fields = [
    "request_latitude",
    "request_longitude",
    "rescue_team_latitude",
    "rescue_team_longitude",
    "dispatch_time",
    "actual_arrival_time",
    "actual_travel_minutes",
    "route_distance_km",
    "estimated_travel_minutes",
    "estimated_arrival_time",
]

for col in eta_fields:
    if col not in eta.columns:
        eta[col] = np.nan

# Preserve the SOS request ID
eta["request_id"] = eta["sos_id"].astype("string")

# 7. Quality checks
if eta["request_id"].duplicated().any():
    raise ValueError(
        "Duplicate SOS request IDs found after merge."
    )

print("\nFinal dataset shape:", eta.shape)
print("Unique SOS requests:", eta["request_id"].nunique())
print(
    "Requests with flood context:",
    int(eta["flood_context_matched"].sum())
)
print(
    "Requests without matched flood context:",
    int((eta["flood_context_matched"] == 0).sum())
)

# 8. Save the consolidated dataset
eta.to_csv(OUTPUT_PATH, index=False)

print("\nSaved successfully:")
print(OUTPUT_PATH)

print("\nFinal preview:")
display(eta.head())

print("\nETA target availability:")
print(
    "Actual travel-time labels:",
    eta["actual_travel_minutes"].notna().sum()
)


SOS shape: (5016, 40)
Flood context shape: (4548, 127)

Flood-context matching:
_flood_join
both          5000
left_only       16
right_only       0
Name: count, dtype: int64

Final dataset shape: (5016, 179)
Unique SOS requests: 5016
Requests with flood context: 5000
Requests without matched flood context: 16


C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\1550799389.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eta = sos.merge(
C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\1550799389.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eta = sos.merge(
C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\1550799389.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis


Saved successfully:
C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv

Final preview:


,sos_id,sos_description,people_affected,people_trapped,injuries_reported,medical_emergency,water_level,building_condition,rescue_required,food_required,...,request_longitude,rescue_team_latitude,rescue_team_longitude,dispatch_time,actual_arrival_time,actual_travel_minutes,route_distance_km,estimated_travel_minutes,estimated_arrival_time,request_id
0,SOS0001,Four people trapped on a roof as water rises r...,4,Yes,No,No,Rapidly_Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOS0001
1,SOS0002,An elderly person is unconscious in a flooded ...,1,Yes,Yes,Yes,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOS0002
2,SOS0003,A child is trapped in fast-moving floodwater.,1,Yes,No,No,Rapidly_Rising,Unknown,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOS0003
3,SOS0004,Several people are trapped inside a collapsing...,6,Yes,Yes,No,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOS0004
4,SOS0005,Water is entering the ground floor and a famil...,5,No,No,No,Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOS0005



ETA target availability:
Actual travel-time labels: 0


In [9]:

from pathlib import Path
import pandas as pd

BASE = Path(r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data")
ETA_DIR = BASE / "ETA_dataset_model"

eta = pd.read_csv(
    ETA_DIR / "eta_dataset_final.csv",
    low_memory=False
)

assistance_path = (
    BASE / "Assitance_Dataset_model"
    / "assistance_model_dataset_final_cleaned.csv"
)

ifi_path = (
    BASE / "ifi_processed_Data"
    / "IFI_flood_events_processed.csv"
)

district_path = (
    BASE / "ifi_processed_Data"
    / "District_FloodImpact_processed.csv"
)

print("FINAL ETA DATASET")
print("Shape:", eta.shape)
print("Missing actual travel-time labels:",
      eta["actual_travel_minutes"].isna().sum())

# Check Assistance linkage without merging
if assistance_path.exists():
    assistance = pd.read_csv(assistance_path, low_memory=False)

    print("\nASSISTANCE DATASET")
    print("Shape:", assistance.shape)
    print("Columns:", assistance.columns.tolist())

    if (
        "disaster_response_message_id_context" in eta.columns
        and "id" in assistance.columns
    ):
        sos_ids = set(
            eta["disaster_response_message_id_context"]
            .dropna().astype(str).str.strip()
        )
        assistance_ids = set(
            assistance["id"].dropna().astype(str).str.strip()
        )

        # Exclude blank and textual missing values
        missing_values = {"", "nan", "none", "<na>"}
        sos_ids -= missing_values
        assistance_ids -= missing_values

        print("Distinct SOS context message IDs:", len(sos_ids))
        print("Distinct Assistance IDs:", len(assistance_ids))
        print("Exact ID overlap:", len(sos_ids & assistance_ids))
    else:
        print("Required message-ID columns are not both available.")

# Inspect IFI files; do not join on uncertain district names
for name, path in [
    ("IFI flood events", ifi_path),
    ("District flood impact", district_path),
]:
    print(f"\n{name.upper()}")

    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
    else:
        print("File not found:", path)

print("\nCheck complete. No files were modified.")


FINAL ETA DATASET
Shape: (5016, 179)
Missing actual travel-time labels: 5016

ASSISTANCE DATASET
Shape: (23753, 12)
Columns: ['id', 'message', 'genre', 'food', 'water', 'shelter', 'medical_help', 'medical_products', 'search_and_rescue', 'transport', 'source_split', 'source_file']
Distinct SOS context message IDs: 4440
Distinct Assistance IDs: 23753
Exact ID overlap: 0

IFI FLOOD EVENTS
Shape: (6876, 17)
Columns: ['Unnamed: 0', 'UEI', 'Start Date', 'End Date', 'Duration(Days)', 'Main Cause', 'Districts', 'State', 'Human fatality', 'Human injured', 'Human Displaced', 'Animal Fatality', 'Description of Casualties/injured', 'Extent of damage', 'Event Source', 'District_LGD_Codes', 'State_Codes']

DISTRICT FLOOD IMPACT
Shape: (732, 5)
Columns: ['Dist_Name', 'Human_fatality', 'Human_injured', 'Population', 'Mean_Flood_Duration']

Check complete. No files were modified.


In [10]:

from pathlib import Path
import pandas as pd
import re
import unicodedata

BASE = Path(r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data")
ETA_DIR = BASE / "ETA_dataset_model"

eta = pd.read_csv(
    ETA_DIR / "eta_dataset_final.csv",
    low_memory=False
)

district_path = (
    BASE / "ifi_processed_Data"
    / "District_FloodImpact_processed.csv"
)

district = pd.read_csv(district_path, low_memory=False)

# Find the SOS district-context column
district_candidates = [
    c for c in eta.columns
    if "district" in c.lower()
    and ("ifi_impact" in c.lower() or "district_context" in c.lower())
]

print("Candidate SOS district columns:", district_candidates)
print("District impact columns:", district.columns.tolist())

def normalize_district(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().strip()
    value = re.sub(r"\bdistrict\b", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

district["district_key"] = district["Dist_Name"].map(
    normalize_district
)

# Duplicate keys must be inspected before using a district lookup.
duplicate_keys = district[
    district["district_key"].duplicated(keep=False)
].sort_values("district_key")

print("\nDuplicate district keys:", len(duplicate_keys))
if not duplicate_keys.empty:
    print(duplicate_keys[["Dist_Name", "district_key"]].head(20))

if district_candidates:
    sos_col = district_candidates[0]

    sos_keys = eta[sos_col].map(normalize_district)
    district_keys = set(district["district_key"]) - {""}

    matched = sos_keys.ne("") & sos_keys.isin(district_keys)

    print("\nDistrict column tested:", sos_col)
    print("SOS rows with district context:", int(sos_keys.ne("").sum()))
    print("Rows with exact normalized district match:", int(matched.sum()))
    print("Unique matched district names:", sos_keys[matched].nunique())

    print("\nExamples of unmatched district names:")
    print(
        eta.loc[
            sos_keys.ne("") & ~sos_keys.isin(district_keys),
            sos_col
        ].drop_duplicates().head(20).to_list()
    )
else:
    print(
        "\nNo suitable district-context column was found. "
        "Inspect the printed columns before proceeding."
    )


Candidate SOS district columns: ['ifi_impact_district_context', 'ifi_area_district_context', 'ifi_severity_district_context']
District impact columns: ['Dist_Name', 'Human_fatality', 'Human_injured', 'Population', 'Mean_Flood_Duration']

Duplicate district keys: 20
           Dist_Name district_key
114       Aurangabad   aurangabad
401       Aurangabad   aurangabad
138        Balrampur    balrampur
656        Balrampur    balrampur
222         Bilaspur     bilaspur
141         Bilaspur     bilaspur
146             East         east
542    East District         east
221         Hamirpur     hamirpur
657         Hamirpur     hamirpur
539  North  District        north
143            North        north
538       Pratapgarh   pratapgarh
658       Pratapgarh   pratapgarh
402          Raigarh      raigarh
140          Raigarh      raigarh
150            South        south
541   South District        south
145             West         west
540    West District         west

District column tes

In [11]:

from pathlib import Path
import pandas as pd
import re
import unicodedata

BASE = Path(r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data")
ETA_DIR = BASE / "ETA_dataset_model"

ETA_PATH = ETA_DIR / "eta_dataset_final.csv"
DISTRICT_PATH = (
    BASE / "ifi_processed_Data"
    / "District_FloodImpact_processed.csv"
)
OUTPUT_PATH = ETA_DIR / "eta_dataset_with_district_features.csv"

# 1. Load datasets
eta = pd.read_csv(ETA_PATH, low_memory=False)
district = pd.read_csv(DISTRICT_PATH, low_memory=False)

# 2. Normalize district names consistently
def normalize_district(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.lower().strip()
    value = re.sub(r"\bdistrict\b", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

eta["_district_key"] = eta[
    "ifi_impact_district_context"
].map(normalize_district)

district["_district_key"] = district[
    "Dist_Name"
].map(normalize_district)

# 3. Convert impact fields to numeric
impact_cols = [
    "Human_fatality",
    "Human_injured",
    "Population",
    "Mean_Flood_Duration",
]

for col in impact_cols:
    district[col] = pd.to_numeric(
        district[col], errors="coerce"
    )

# 4. Audit duplicate keys before aggregation
duplicate_rows = district[
    district["_district_key"].duplicated(keep=False)
].sort_values("_district_key")

print("Duplicate-key source rows:", len(duplicate_rows))
print("Duplicate district keys:",
      duplicate_rows["_district_key"].nunique())

# Inspect the conflicting source rows before accepting aggregation.
print("\nDuplicate-key details:")
print(
    duplicate_rows[
        ["Dist_Name", "_district_key"] + impact_cols
    ].to_string(index=False)
)

# 5. Create one lookup row per normalized district.
# Mean is used only to prevent duplicate-key row multiplication.
district_lookup = (
    district[district["_district_key"].ne("")]
    .groupby("_district_key", as_index=False)[impact_cols]
    .mean()
)

# 6. Rename added columns so their source is explicit
district_lookup = district_lookup.rename(columns={
    col: f"district_impact_{col.lower()}"
    for col in impact_cols
})

# 7. Merge without multiplying SOS records
result = eta.merge(
    district_lookup,
    on="_district_key",
    how="left",
    validate="many_to_one",
    indicator="_district_join",
)

result["district_impact_matched"] = (
    result["_district_join"] == "both"
).astype("int8")

print("\nDistrict matching:")
print(result["_district_join"].value_counts())

result.drop(
    columns=["_district_key", "_district_join"],
    inplace=True
)

# 8. Validate and save
assert len(result) == len(eta), "SOS row count changed!"
assert result["request_id"].is_unique, "Duplicate request IDs!"

result.to_csv(OUTPUT_PATH, index=False)

print("\nOriginal shape:", eta.shape)
print("New shape:", result.shape)
print("Unique SOS requests:", result["request_id"].nunique())
print(
    "District feature matches:",
    int(result["district_impact_matched"].sum())
)
print("Saved to:", OUTPUT_PATH)


C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\1483357206.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  eta["_district_key"] = eta[
C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\1483357206.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = eta.merge(
C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\1483357206.py:84: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using 

Duplicate-key source rows: 20
Duplicate district keys: 10

Duplicate-key details:
      Dist_Name _district_key  Human_fatality  Human_injured  Population  Mean_Flood_Duration
     Aurangabad    aurangabad             301             46     3160597                 13.0
     Aurangabad    aurangabad             301             46     4525057                 13.0
      Balrampur     balrampur              58             26      853775                 20.0
      Balrampur     balrampur              58             26     2570002                 20.0
       Bilaspur      bilaspur             111              9      517758                  5.0
       Bilaspur      bilaspur             111              9     2180689                  5.0
           East          east               0              0     1368632                  NaN
  East District          east             146              8      337891                  2.0
       Hamirpur      hamirpur             219             20      502514

In [12]:

from pathlib import Path
import pandas as pd

ETA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

source = ETA_DIR / "eta_dataset_with_district_features.csv"
output = ETA_DIR / "eta_dataset_consolidated.csv"

df = pd.read_csv(source, low_memory=False)

# Remove provisional district lookup values because duplicate
# district names have not yet been resolved geographically.
provisional_cols = [
    "district_impact_human_fatality",
    "district_impact_human_injured",
    "district_impact_population",
    "district_impact_mean_flood_duration",
    "district_impact_matched",
]

df = df.drop(
    columns=[c for c in provisional_cols if c in df.columns]
).copy()

# Validate the final consolidated feature file.
assert len(df) == 5016
assert df["request_id"].is_unique

df.to_csv(output, index=False)

print("Final consolidated dataset saved.")
print("Shape:", df.shape)
print("Unique SOS requests:", df["request_id"].nunique())
print("Output:", output)
print("\nActual travel-time labels available:",
      df["actual_travel_minutes"].notna().sum())


Final consolidated dataset saved.
Shape: (5016, 179)
Unique SOS requests: 5016
Output: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_consolidated.csv

Actual travel-time labels available: 0


In [13]:

from pathlib import Path
import pandas as pd
import re

# Paths
ETA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

INPUT = ETA_DIR / "eta_dataset_consolidated.csv"
OUTPUT = ETA_DIR / "eta_dataset_final.csv"

# 1. Load consolidated dataset
df = pd.read_csv(INPUT, low_memory=False)
initial_rows = len(df)

# 2. Clean column names
def clean_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df.columns = [clean_column_name(c) for c in df.columns]

# 3. Remove accidental index columns
df = df.loc[
    :,
    ~df.columns.str.match(r"^unnamed(:_\d+)?$|^index$")
].copy()

# 4. Remove duplicate columns, keeping the first occurrence
df = df.loc[:, ~df.columns.duplicated()].copy()

# 5. Clean text values without changing meaningful labels
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

# 6. Remove completely empty columns, except the ETA target
# Keep actual_travel_minutes to explicitly show the missing target.
target = "actual_travel_minutes"
empty_cols = [
    col for col in df.columns
    if df[col].isna().all() and col != target
]
df = df.drop(columns=empty_cols).copy()

# 7. Remove duplicate SOS requests, if any
if "request_id" not in df.columns:
    raise ValueError("request_id column is missing.")

df = df.drop_duplicates(subset="request_id", keep="first").copy()

# 8. Validate
assert df["request_id"].notna().all(), "Missing request IDs."
assert df["request_id"].is_unique, "Duplicate request IDs remain."
assert len(df) > 0, "Dataset is empty."

# 9. Save final cleaned dataset
df.to_csv(OUTPUT, index=False)

print("FINAL CLEANED ETA DATASET")
print("Original rows:", initial_rows)
print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("Unique SOS requests:", df["request_id"].nunique())
print("Duplicate request IDs:", df["request_id"].duplicated().sum())
print("Missing values:", int(df.isna().sum().sum()))
print("Actual travel-time labels available:",
      df[target].notna().sum() if target in df.columns else 0)
print("\nSaved to:", OUTPUT)
print("\nPreview:")
display(df.head())


FINAL CLEANED ETA DATASET
Original rows: 5016
Final rows: 5016
Final columns: 170
Unique SOS requests: 5016
Duplicate request IDs: 0
Missing values: 78149
Actual travel-time labels available: 0

Saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv

Preview:


,sos_id,sos_description,people_affected,people_trapped,injuries_reported,medical_emergency,water_level,building_condition,rescue_required,food_required,...,1995_hdi,1990_hdi,population_count,population_density,land_cover,soil_type,lithology_type,flood_context_matched,actual_travel_minutes,request_id
0,SOS0001,Four people trapped on a roof as water rises r...,4,Yes,No,No,Rapidly_Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,0,NaN,SOS0001
1,SOS0002,An elderly person is unconscious in a flooded ...,1,Yes,Yes,Yes,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,0,NaN,SOS0002
2,SOS0003,A child is trapped in fast-moving floodwater.,1,Yes,No,No,Rapidly_Rising,Unknown,Yes,No,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,0,NaN,SOS0003
3,SOS0004,Several people are trapped inside a collapsing...,6,Yes,Yes,No,Rising,Damaged,Yes,No,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,0,NaN,SOS0004
4,SOS0005,Water is entering the ground floor and a famil...,5,No,No,No,Rising,Intact,Yes,No,...,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,0,NaN,SOS0005


In [14]:

from pathlib import Path
import pandas as pd

path = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv"
)

df = pd.read_csv(path, low_memory=False)

# Missing-value report
report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percent": (
        df.isna().mean().values * 100
    ).round(2),
    "unique_values": df.nunique(dropna=True).values
})

report = report.sort_values(
    ["missing_percent", "column"],
    ascending=[False, True]
)

print("Dataset shape:", df.shape)
print("Rows:", len(df))
print("Duplicate request IDs:", df["request_id"].duplicated().sum())

print("\nColumns with 100% missing values:")
print(
    report.loc[
        report["missing_percent"] == 100,
        ["column", "missing_count"]
    ].to_string(index=False)
)

print("\nColumns with more than 50% missing values:")
print(
    report.loc[
        report["missing_percent"] > 50,
        ["column", "missing_count", "missing_percent"]
    ].to_string(index=False)
)

print("\nMissing-value summary:")
print("Total missing cells:", int(df.isna().sum().sum()))
print("Actual travel-time labels:", df["actual_travel_minutes"].notna().sum())


Dataset shape: (5016, 170)
Rows: 5016
Duplicate request IDs: 0

Columns with 100% missing values:
               column  missing_count
actual_travel_minutes           5016

Columns with more than 50% missing values:
                          column  missing_count  missing_percent
           actual_travel_minutes           5016           100.00
      no_of_seventhorder_streams           5005            99.78
     seventhorder_streams_length           5005            99.78
seventhorder_streams_mean_length           5005            99.78
  sixthseventh_bifurcation_ratio           5005            99.78
sixthseventh_stream_length_ratio           5005            99.78
    fifthsixth_bifurcation_ratio           4260            84.93
  fifthsixth_stream_length_ratio           4260            84.93
        no_of_sixthorder_streams           4260            84.93
       sixthorder_streams_length           4260            84.93
  sixthorder_streams_mean_length           4260            84.93
    

In [15]:

from pathlib import Path
import pandas as pd

ETA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

FILE = ETA_DIR / "eta_dataset_final.csv"

df = pd.read_csv(FILE, low_memory=False)

# 1. Remove columns with no usable information
empty_cols = [
    c for c in df.columns
    if df[c].isna().all()
]
df = df.drop(columns=empty_cols).copy()

# 2. Keep the request identifier; do not use it as a model feature
if "request_id" not in df.columns:
    df["request_id"] = df["sos_id"].astype("string")

# 3. Fill numeric feature missing values using medians
numeric_cols = df.select_dtypes(include="number").columns.tolist()

for col in numeric_cols:
    if col != "request_id" and df[col].notna().any():
        df[col] = df[col].fillna(df[col].median())

# 4. Fill missing categorical/text values
categorical_cols = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in categorical_cols:
    if col not in ["request_id", "sos_id"]:
        df[col] = df[col].fillna("Unknown")

# 5. Validate identifiers and remaining missing values
assert len(df) == 5016
assert df["request_id"].is_unique

# 6. Save to the requested final filename
df.to_csv(FILE, index=False)

print("FINAL FEATURE DATASET")
print("Shape:", df.shape)
print("Unique SOS requests:", df["request_id"].nunique())
print("Remaining missing cells:", int(df.isna().sum().sum()))
print("Completely empty columns removed:", empty_cols)
print("Saved:", FILE)


FINAL FEATURE DATASET
Shape: (5016, 169)
Unique SOS requests: 5016
Remaining missing cells: 0
Completely empty columns removed: ['actual_travel_minutes']
Saved: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv


In [16]:

from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------------
# JEEVAN LINK: SYNTHETIC ETA TARGET GENERATION
# --------------------------------------------------

ETA_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

FILE = ETA_DIR / "eta_dataset_final.csv"
OUTPUT = ETA_DIR / "eta_dataset_final.csv"

df = pd.read_csv(FILE, low_memory=False)
rng = np.random.default_rng(42)

# 1. Find route-distance column, if one exists
distance_candidates = [
    "route_distance_km",
    "distance_km",
    "road_distance_km",
    "distance"
]

distance_col = next(
    (c for c in distance_candidates if c in df.columns),
    None
)

if distance_col:
    distance = pd.to_numeric(df[distance_col], errors="coerce")
    distance = distance.where(distance > 0)
else:
    distance = pd.Series(np.nan, index=df.index)

# 2. Simulate distance only when real route distance is unavailable.
# This is a simulation assumption, not measured geography.
simulated_distance = pd.Series(
    rng.uniform(2, 20, len(df)),
    index=df.index
)
distance = distance.fillna(simulated_distance)

# 3. Estimate baseline travel time.
# Assumed average speed: 25 km/h.
base_minutes = distance / 25 * 60

# 4. Add urgency adjustment
urgency_col = next(
    (c for c in df.columns if c.lower() == "urgency_label"),
    None
)

if urgency_col:
    urgency = df[urgency_col].astype("string").str.lower()
else:
    urgency = pd.Series("medium", index=df.index)

urgency_adjustment = urgency.map({
    "critical": -2,
    "high": -1,
    "medium": 0,
    "low": 2
}).fillna(0)

# 5. Add a flood-condition delay
flood_delay = pd.Series(0.0, index=df.index)

if "water_level" in df.columns:
    water = df["water_level"].astype("string").str.lower()
    flood_delay += np.where(
        water.str.contains("rapid|rising", na=False), 5, 0
    )

if "building_condition" in df.columns:
    building = df["building_condition"].astype("string").str.lower()
    flood_delay += np.where(
        building.str.contains("damaged|collaps", na=False), 3, 0
    )

# 6. Generate positive, bounded synthetic ETA labels
noise = rng.normal(loc=0, scale=2, size=len(df))

synthetic_eta = (
    base_minutes
    + urgency_adjustment
    + flood_delay
    + noise
).clip(3, 120)

df["actual_travel_minutes"] = np.round(synthetic_eta, 1)

# Mark the provenance clearly
df["eta_label_source"] = "synthetic_simulation"
df["eta_is_synthetic"] = 1

# Store the simulation assumptions for transparency
df["eta_assumed_average_speed_kmh"] = 25.0

# 7. Validate
assert len(df) == 5016
assert df["request_id"].is_unique
assert df["actual_travel_minutes"].notna().all()
assert df["actual_travel_minutes"].between(3, 120).all()

# 8. Save
df.to_csv(OUTPUT, index=False)

print("SYNTHETIC ETA DATASET SAVED")
print("Shape:", df.shape)
print("Unique SOS requests:", df["request_id"].nunique())
print("Missing ETA targets:", df["actual_travel_minutes"].isna().sum())
print("Label source:", df["eta_label_source"].unique())
print("\nSynthetic ETA distribution:")
print(df["actual_travel_minutes"].describe())
print("\nSaved to:", OUTPUT)


C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\701136417.py:93: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["actual_travel_minutes"] = np.round(synthetic_eta, 1)
C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\701136417.py:96: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["eta_label_source"] = "synthetic_simulation"
C:\Users\ishit\AppData\Local\Temp\ipykernel_18204\701136417.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

SYNTHETIC ETA DATASET SAVED
Shape: (5016, 173)
Unique SOS requests: 5016
Missing ETA targets: 0
Label source: <StringArray>
['synthetic_simulation']
Length: 1, dtype: str

Synthetic ETA distribution:
count    5016.000000
mean       29.718062
std        12.902169
min         3.000000
25%        18.800000
50%        29.600000
75%        40.700000
max        58.300000
Name: actual_travel_minutes, dtype: float64

Saved to: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv


In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import shutil

folder = Path(
    r"C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model"
)

original = folder / "eta_dataset_final.csv"
backup = folder / "eta_dataset_final_backup_before_feature_reduction.csv"

# 1. Read the original dataset and preserve a backup
df = pd.read_csv(original, low_memory=False)

if not backup.exists():
    shutil.copy2(original, backup)

# 2. Select ETA-relevant columns that exist in your dataset
candidate_features = [
    "request_id",
    "urgency_label",
    "people_affected",
    "people_trapped",
    "injuries_reported",
    "medical_emergency",
    "water_level",
    "building_condition",
    "rescue_required",
    "location_risk",
    "context_peak_flood_level_m",
    "context_event_duration_days",
    "context_rainfall_t1d",
    "context_percent_flooded_area",
    "context_dfsi",
    "flood_type",
    "peak_discharge_q_cumec",
    "flood_volume_cumec",
    "time_to_peak_days",
    "recession_time_day",
    "drainage_area",
    "catchment_relief",
    "drainage_density",
    "annual_precipitation",
    "road_density",
    "population_density",
    "urban_percentage",
]

selected = [c for c in candidate_features if c in df.columns]
eta = df[selected].copy()

if "request_id" not in eta.columns:
    raise ValueError("request_id is missing; original file was not changed.")

# 3. Clean selected feature values
for col in eta.select_dtypes(include="number").columns:
    eta[col] = eta[col].replace([np.inf, -np.inf], np.nan)
    if eta[col].notna().any():
        eta[col] = eta[col].fillna(eta[col].median())

for col in eta.select_dtypes(
    include=["object", "string", "category"]
).columns:
    eta[col] = eta[col].fillna("Unknown").astype(str).str.strip()
    eta[col] = eta[col].replace("", "Unknown")

# 4. Preserve existing synthetic ETA labels when available.
# Generate labels only if the original target is absent.
if "actual_travel_minutes" in df.columns:
    eta["actual_travel_minutes"] = pd.to_numeric(
        df["actual_travel_minutes"], errors="coerce"
    )
else:
    rng = np.random.default_rng(42)
    distance = rng.uniform(2, 20, len(df))
    urgency = (
        eta["urgency_label"].astype(str).str.lower()
        if "urgency_label" in eta.columns
        else pd.Series("medium", index=eta.index)
    )
    adjustment = urgency.map({
        "critical": -2, "high": -1, "medium": 0, "low": 2
    }).fillna(0)
    eta["actual_travel_minutes"] = np.round(
        (distance / 25 * 60 + adjustment
         + rng.normal(0, 2, len(df))).clip(3, 120),
        1
    )

eta["eta_label_source"] = "synthetic_simulation"
eta["eta_is_synthetic"] = 1

# 5. Validate before overwriting the original
assert len(eta) == len(df) == 5016
assert eta["request_id"].is_unique
assert eta["actual_travel_minutes"].notna().all()

# 6. Replace original file with the reduced dataset
eta.to_csv(original, index=False)

# 7. Verify saved file
check = pd.read_csv(original, low_memory=False)

print("Original final dataset updated successfully.")
print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Unique requests:", check["request_id"].nunique())
print("Missing cells:", int(check.isna().sum().sum()))
print("Backup:", backup)
print("Updated file:", original)


Original final dataset updated successfully.
Rows: 5016
Columns: 30
Unique requests: 5016
Missing cells: 0
Backup: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final_backup_before_feature_reduction.csv
Updated file: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\ETA_dataset_model\eta_dataset_final.csv
